This notebook walks through the process of constructing a similarity graph using the axelrodVoice package.

In [ ]:
import axelrodVoice as axlV
import pandas as pd

Define the subject and opponent strategies. Subjects are graphed based on their matches with opponents.

In [ ]:
subjects = [s() for s in [axlV.Cooperator, axlV.Defector, axlV.Grudger, axlV.TitForTat]]
opponents = [o() for o in [axlV.Cooperator, axlV.Defector, axlV.Liar, axlV.TestD7]]

Generate data from the matches, then create a graph object using that data.

In [ ]:
axlV.SimilarityGraph.generate_data(subjects, opponents, last_round=100, filename="small_data.csv", seeds=[1])

In [ ]:
graph = axlV.SimilarityGraph("small_data.csv")

In [ ]:
behaviors = ["per_cC", "per_cD", "per_dC", "per_dD"]
(graph.full_data.sort_values(by=['opponent', 'seed', 'subject'])
         .loc[:, ["opponent", "subject"] + behaviors]
         .reset_index(drop=True)
         .assign(max_val= lambda df: df.max(axis=1,numeric_only=True),
                 max_col= lambda df: df.idxmax(axis=1,numeric_only=True))
         .assign(val = lambda df: df.apply(lambda row: f"{row['max_val']:.0%} {row['max_col'][4:]}", axis=1))
         .loc[:, ["opponent", "subject", "val"]]
         .pivot(index="subject", columns="opponent", values="val")
         )
# TODO: investigate why COOP only 99% cC

In [ ]:
# Create clusters using behaviors
s_names = [s.name for s in subjects]
o_names = [o.name for o in opponents]

graph.partition_by_metric_and_threshold(
    behaviors=behaviors,
    threshold=0.15,
    subjects=s_names, opponents=o_names,
    start=1, end=100)

In [ ]:
# Figure 5.1.2a
adj_mat = graph.partitions_to_freq_tbl()
custom_adj = pd.DataFrame(0.0, index=s_names, columns=s_names)
custom_adj[ custom_adj.index == 'TIFT' ] = adj_mat[ adj_mat.index == 'TIFT' ]
graph.draw_similarity_graph(adj_matrix=custom_adj)

In [ ]:
# Figure 5.1.2b
custom_adj = pd.DataFrame(0.0, index=s_names, columns=s_names)
custom_adj['TIFT'] = adj_mat['TIFT']
graph.draw_similarity_graph(adj_matrix=custom_adj)

In [ ]:
# Figure 5.1.2c
graph.draw_similarity_graph()